# Dummy Classifier — v2

**Van Nguyen Thanh Dat — 3124411067 · Nhóm 10 · UCI 697**

## Mục tiêu thí nghiệm

So sánh hai chiến lược dự đoán trên cùng 5 fold cố định của nhóm.

- **v1 (`most_frequent`)** luôn dự đoán lớp phổ biến nhất trong
  phần huấn luyện của mỗi fold.
- **v2 (`stratified`)** lấy mẫu nhãn dự đoán theo tỷ lệ các lớp
  trong phần huấn luyện của mỗi fold, với `random_state=42`.

So sánh bằng Macro-F1 trung bình trên 5 fold và dùng kết quả OOF
để phân tích từng lớp. Run này chưa chấm tập Test.

In [1]:
from pathlib import Path
import sys


current_dir = Path.cwd()
search_dirs = (current_dir, *current_dir.parents)

FINAL = next(
    folder
    for folder in search_dirs
    if (folder / "src" / "data.py").is_file()
    and (folder / "data" / "raw" / "data.csv").is_file()
)

sys.path.insert(0, str(FINAL))

from src.data import LABELS, load_student_data

print("Thư mục dự án:", FINAL)

Thư mục dự án: c:\Users\Dat\Documents\Môn đang học\Máy học\Đồ án\final_codespaces\final


## Mục 2 - Dữ liệu và nhãn

In [2]:
from IPython.display import display


X, y = load_student_data()

print("Số mẫu:", len(y))
print("Số đặc trưng:", X.shape[1])

label_counts = y.value_counts()
label_counts = label_counts.reindex(LABELS)
label_summary = label_counts.rename_axis("Nhãn").to_frame("Số mẫu")

display(label_summary)

Số mẫu: 4424
Số đặc trưng: 36


,Số mẫu
Nhãn,
Dropout,1421
Enrolled,794
Graduate,2209


## Mục 3 - Nạp bộ chia Train/Test và 5 fold

In [3]:
from src.data import load_splits


train_ids, test_ids, folds, split_manifest = load_splits()

X_train = X.iloc[train_ids]
y_train = y.iloc[train_ids]

print("Số mẫu Train:", len(y_train))
print("Số mẫu Test giữ riêng:", len(test_ids))
print("Số fold:", len(folds))

for fold_number, (fit_positions, valid_positions) in enumerate(folds, start=1):
    print(
        f"Fold {fold_number}: "
        f"{len(fit_positions)} mẫu huấn luyện, "
        f"{len(valid_positions)} mẫu validation"
    )

Số mẫu Train: 3539
Số mẫu Test giữ riêng: 885
Số fold: 5
Fold 1: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 2: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 3: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 4: 2831 mẫu huấn luyện, 708 mẫu validation
Fold 5: 2832 mẫu huấn luyện, 707 mẫu validation


## Mục 4 - Đọc cấu hình và tạo Dummy Classifier v2

In [4]:
from runpy import run_path

import yaml


MODEL_ID = "m01_baseline"
MODEL_VERSION = "v2"

model_dir = FINAL / "models" / MODEL_ID / MODEL_VERSION
config_path = model_dir / "config.yaml"
train_path = model_dir / "train.py"

config_text = config_path.read_text(encoding="utf-8")
config = yaml.safe_load(config_text)

assert config["model_id"] == MODEL_ID
assert config["version"] == MODEL_VERSION

training_module = run_path(str(train_path))
make_model = training_module["make_model"]
model = make_model(config)

print("Mô hình:", model)
display(config["params"])

Mô hình: DummyClassifier(random_state=42, strategy='stratified')


{'strategy': 'stratified', 'random_state': 42}

## Mục 5 - Chọn run Dummy Classifier v2 đã lưu

In [5]:
RUN_ID = "m01_baseline_v2_20261105T032722_ca767d56"
run_path = FINAL / "experiments" / "runs" / RUN_ID

if not run_path.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục kết quả: {run_path}"
    )

print("Run:", RUN_ID)
print("Đường dẫn:", run_path.relative_to(FINAL).as_posix())

Run: m01_baseline_v2_20261105T032722_ca767d56
Đường dẫn: experiments/runs/m01_baseline_v2_20261105T032722_ca767d56


## Mục 6 - Đọc kết quả CV

In [6]:
import json

import pandas as pd


metrics_path = run_path / "metrics.json"
metrics_text = metrics_path.read_text(encoding="utf-8")
metrics = json.loads(metrics_text)

cv = metrics["cv"]
macro_f1_mean = cv["valid_f1_macro_mean"]
macro_f1_std = cv["valid_f1_macro_std"]
accuracy_mean = cv["valid_accuracy_mean"]
dropout_recall_mean = cv["valid_recall_Dropout_mean"]

print(f"Macro-F1 CV: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Accuracy CV: {accuracy_mean:.2%}")
print(f"Recall Dropout CV: {dropout_recall_mean:.2%}")

fold_metrics_path = run_path / "fold_metrics.csv"
fold_metrics = pd.read_csv(fold_metrics_path)

columns = [
    "fold",
    "train_f1_macro",
    "valid_f1_macro",
    "valid_accuracy",
    "valid_recall_Dropout",
    "gap_f1_macro",
]

display(fold_metrics[columns].set_index("fold").round(4))

Macro-F1 CV: 0.3383 ± 0.0092
Accuracy CV: 39.19%
Recall Dropout CV: 34.65%


,train_f1_macro,valid_f1_macro,valid_accuracy,valid_recall_Dropout,gap_f1_macro
fold,,,,,
1,0.3275,0.3316,0.3969,0.3744,-0.0041
2,0.3323,0.3429,0.3941,0.3304,-0.0105
3,0.3335,0.3507,0.3997,0.3333,-0.0172
4,0.3264,0.3244,0.3757,0.3377,0.0021
5,0.3325,0.3417,0.3932,0.3568,-0.0092


## Mục 7 - Kết quả OOF và ma trận nhầm lẫn

Gộp dự đoán trên phần validation của 5 fold để xem Dummy v2
dự đoán đúng và nhầm bao nhiêu mẫu ở từng lớp.

In [7]:
class_metrics_path = run_path / "class_metrics_oof.csv"
class_metrics = pd.read_csv(class_metrics_path)
class_summary = class_metrics.set_index("label").reindex(LABELS)

macro_f1_oof = metrics["oof"]["f1_macro"]
accuracy_oof = metrics["oof"]["accuracy"]

print(f"Macro-F1 OOF: {macro_f1_oof:.4f}")
print(f"Accuracy OOF: {accuracy_oof:.2%}")
display(class_summary.round(4))

confusion_matrix_path = run_path / "confusion_matrix.csv"
confusion_matrix = pd.read_csv(confusion_matrix_path, index_col=0)
confusion_matrix = confusion_matrix.loc[list(LABELS), list(LABELS)]

print("Hàng là nhãn thật, cột là nhãn dự đoán.")
display(confusion_matrix)

Macro-F1 OOF: 0.3383
Accuracy OOF: 39.19%


,precision,recall,f1-score,support
label,,,,
Dropout,0.3300,0.3465,0.3381,1137.0
Enrolled,0.1828,0.1669,0.1745,635.0
Graduate,0.5025,0.5020,0.5023,1767.0


Hàng là nhãn thật, cột là nhãn dự đoán.


,Dropout,Enrolled,Graduate
true_label,,,
Dropout,394,183,560
Enrolled,211,106,318
Graduate,589,291,887


## Mục 8 - Truy vết lần chạy

In [8]:
git_info_path = run_path / "git_info.json"
git_info_text = git_info_path.read_text(encoding="utf-8")
git_info = json.loads(git_info_text)

base_commit = git_info.get("base_commit") or "Không ghi nhận"
test_scored = metrics["test_scored_this_run"]

print("Run:", RUN_ID)
print("Commit gốc ghi trong run:", base_commit)
print("Đã chấm tập Test:", "Có" if test_scored else "Chưa")

Run: m01_baseline_v2_20261105T032722_ca767d56
Commit gốc ghi trong run: 9c8d3b665d4f6e771005d640363d90b984934612
Đã chấm tập Test: Chưa


## Kết luận

Trên cùng 5 fold, Dummy v2 (`stratified`) đạt Macro-F1 CV
**0,3383 ± 0,0092**, cao hơn Dummy v1 (`most_frequent`) với
**0,2220 ± 0,0002**. Accuracy CV giảm từ **49,93%** xuống **39,19%**.

Trong dự đoán OOF, v2 nhận đúng **394/1.137** mẫu Dropout,
**106/635** mẫu Enrolled và **887/1.767** mẫu Graduate.
Enrolled có Recall thấp nhất (**16,69%**) và thường bị dự đoán
nhầm sang Graduate (**318 mẫu**).

V1 là mốc luôn dự đoán lớp phổ biến nhất. V2 là mốc dự đoán
ngẫu nhiên theo tỷ lệ lớp. Điểm Macro-F1 cao hơn cho thấy v2
nhận diện được thêm các lớp ít mẫu, với đánh đổi là Accuracy thấp hơn.

Run này chỉ đánh giá CV và OOF trên tập Train. Tập Test chưa được chấm.
Mã `9c8d3b6` trong `git_info.json` thuộc repo Hue Tourism ở thư mục
cha. Commit nguồn của Student Dropout chưa được ghi nhận cho lần chạy này.